In [74]:
import numpy as np
import pandas as pd
import sys
from pathlib import Path

sys.path.append(str(Path("..").resolve()))
from script.load_data import customer_data


In [75]:
customer_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


In [76]:
# customer_data["TotalCharges"] = pd.to_numeric(customer_data["TotalCharges"],errors="coerce")
# print(customer_data["TotalCharges"].isna().sum())

customer_data["TotalCharges"] = customer_data["TotalCharges"].fillna(0)
customer_data["TotalCharges"] = pd.to_numeric(customer_data["TotalCharges"],errors="coerce")


In [77]:
# print(customer_data["TotalCharges"])
# print(customer_data.isna().sum())
print(customer_data["PaymentMethod"].value_counts())

PaymentMethod
Electronic check             2365
Mailed check                 1612
Bank transfer (automatic)    1544
Credit card (automatic)      1522
Name: count, dtype: int64


In [78]:
# Dividing the columns into categorical and numeric 
X = customer_data.drop(columns=["customerID","Churn"])
y = customer_data["Churn"]

numeric_columns = ["tenure", "MonthlyCharges", "TotalCharges"]
categorical_columns = [
    "gender", "SeniorCitizen", "Partner", "Dependents",
    "PhoneService", "MultipleLines", "InternetService",
    "OnlineSecurity", "OnlineBackup", "DeviceProtection",
    "TechSupport", "StreamingTV", "StreamingMovies",
    "Contract", "PaperlessBilling", "PaymentMethod"
]

In [79]:
from sklearn.model_selection import train_test_split

customer_data["Churn"] = customer_data["Churn"].replace({"Yes": 1, "No":0})

X = customer_data.drop(columns=["Churn", "customerID"])
y = customer_data["Churn"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


In [80]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

num_pipeline = Pipeline(
    [("imputer",SimpleImputer(strategy = "median")),
    ("scaler",StandardScaler())]
)

preprocessor = ColumnTransformer(
        transformers = [
        ("num",num_pipeline,numeric_columns),
        ("cat",OneHotEncoder(),categorical_columns)
        ]
)

pipeline = Pipeline([
    ("preprocessor",preprocessor)
])

X_train_transformed = pipeline.fit_transform(X_train)
X_test_transformed = pipeline.transform(X_test)

In [81]:
import numpy as np

y_pred = np.array([0.1, 0.2, 0.03, 0.1, 0.7, 0.8, 0.9])

mask = (y_pred > 0.5).astype(int)

print(mask)


[0 0 0 0 1 1 1]


In [82]:
from sklearn.linear_model import LogisticRegression as SkLr

model = SkLr()

model.fit(X_train_transformed,y_train)

sk_pred = model.predict(X_test_transformed)
print(sk_pred)


[1 0 0 ... 0 0 0]


In [85]:
import os
import sys
sys.path.append(os.path.abspath(".."))
from script.logisticregression import LogisticRegression

In [88]:
model = LogisticRegression(0.01,1000)
model.fit(X_train_transformed, y_train)

y_pred = model.predict(X_test_transformed)
print(y_pred)

[1 0 0 ... 0 0 0]
